In [5]:
from dotenv import load_dotenv 
load_dotenv()

True

In [6]:
from langchain_mistralai import MistralAIEmbeddings,ChatMistralAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter 
import time 
from sentence_transformers import CrossEncoder 

In [7]:
llm_model = ChatMistralAI(
    model = "mistral-small-2603"
)
embedding_model = MistralAIEmbeddings(
)


reranker_model = CrossEncoder(
    "BAAI/bge-reranker-base"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [8]:
from email_validator.syntax import split_email
def generate_hypo_docs(query):
    prompt=f"Generate a hypothetical best-guess answer to: {query}, only the answer of the query is required."
    response = llm_model.invoke(prompt).content
    return response 

def embed_response(response):
    embedded_response = embedding_model.embed_query(response)
    return embedded_response

def load_documents(file_path =r"d:\My-Learning\AdvanceRag\Documents\Indian_constitution.pdf"):
    loader = PyPDFLoader(file_path) 
    documents = loader.load()
    return documents

def text_split(document):
    splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=100)
    chunks = splitter.split_documents(document)
    return chunks

def create_vector_db(embedding_model,store_dir=r"d:\My-Learning\AdvanceRag\Documents\Vector_db"):
    vector_store = Chroma(
        collection_name = "Documents",
        embedding_function = embedding_model,
        persist_directory = store_dir 
    )
    return vector_store

def insert_vector_db(vector_store,chunks):
    try:
        vector_store.add_documents(chunks)
        print("data has been added successfully")
    except:
        print("an error has occured")

def search_vector_db(vector_store,response):
    start_time = time.time()
    retriver = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 25,
        "fetch_k": 40,
        "lambda_mult": 0.7
    }
)
    reterived_docs = retriver.invoke(response)
    endtime = time.time()
    print(f"time taken to reterive the documents : {endtime-start_time}")
    return reterived_docs
    
def rerank_document(docs,k,query):
    """
    This function re-ranks the provided list of queries based on semantic scores.
    
    docs : list of reterived documents.
    k : number of docs as output. 
    query : original query from the user.
    """
    start_time = time.time()

    pairs = [(query,doc.page_content) for doc in docs]

    scores = reranker_model.predict(pairs)

    scores_docs = list(zip(docs,scores))

    sorted_scores = sorted(
        scores_docs,
        key = lambda x : x[1],
        reverse=True 
    )

    reranked_docs = [doc for doc,score in sorted_scores[:k]]
    end_time = time.time()
    print("Time taken for reranking : ",end_time-start_time)
    return reranked_docs

In [9]:
def workflow_store():
    # load file into documents 
    documents = load_documents()
    # split into text
    chunks = text_split(documents)
    # create vector database 
    vector_db = create_vector_db(embedding_model)
    # insert data into vector database 
    insert_vector_db(vector_db,chunks)
    print("Workflow completed succesfully.")
    return vector_db

def retrive_data(query,vector_db):
    #generate hypothetical
    start_time = time.time()
    response = generate_hypo_docs(query)
    # print(response)
    # search the embeded response in the vectordb 
    reterived_docs = search_vector_db(vector_db,response)
    context = rerank_document(reterived_docs,5,query)
    # print(context)
    prompt = f"""
    You are a helpful assistant.
    Your main goal is to answer the user's QUERY from PROVIDED CONTEXT,
    INSTRUCTION : 
    1. analyse the user query
    2. analyse the provided context and talior a concised and simple response for the user.
    3. if the CONTEXT isnt appropriate or isnt related to query just say "I dont know".

    QUERY : {query}
    CONTEXT : {context}
    """
    final_output = llm_model.invoke(prompt)
    print(final_output.content)
    end_time = time.time()
    print(f"time taken to complete the Reterival section : {end_time-start_time}")

In [9]:
vector_store = workflow_store()

data has been added successfully
Workflow completed succesfully.


In [12]:
query = input("ask your question from constitution of India")
vector_store = create_vector_db(embedding_model)
retrive_data(query,vector_store)

time taken to reterive the documents : 0.5964820384979248
Time taken for reranking :  1.8233246803283691
The powers of the **Prime Minister of India**, as per the provided context from the **Indian Constitution**, are:

1. **Head of the Council of Ministers** – The Prime Minister leads the Council of Ministers, which aids and advises the President in the administration of Union affairs (Article 74).
2. **Communication with the President** – The Prime Minister is responsible for:
   - Informing the President about all decisions of the Council of Ministers related to Union administration.
   - Furnishing any information the President may request regarding Union administration or legislative proposals (Article 78).
time taken to complete the Reterival section : 6.406429290771484
